# 彻底拆解多头注意力机制 (Multi-Head Attention)
这个 Notebook 将完整展示输入张量是如何在多头注意力机制中被：
1. 线性映射 (Linear Projection)
2. 切分多头 (Split Heads)
3. 掩码遮挡 (Masking)
4. 归一化打分 (Softmax)
5. 重新拼接合并 (Concatenation) 的全过程！

In [1]:
import torch
import torch.nn as nn
import math

# 设置 PyTorch 打印格式，保留2位小数，关闭科学计数法，让矩阵看起来极其清爽
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

# 固定随机数种子，保证咱们每次跑出来的矩阵数字是一模一样的！
torch.manual_seed(42) 

print("环境准备完毕！PyTorch 打印格式已设置。")

/home/xdwang/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


环境准备完毕！PyTorch 打印格式已设置。


In [2]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0
        self.d_k = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):         
        d_k = query.shape[-1]
        
        # 1. 矩阵乘法打分
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)  
        
        # 2. Mask 操作
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
            
        # 3. Softmax 归一化
        attention_scores = attention_scores.softmax(dim=-1)
        
        print("\n" + "="*40)
        print("【进入核心 Attention 打分环节】")
        print("--- 步骤 4：注意力权重 (Softmax 之后) ---")
        print("形状:", attention_scores.shape, "-> (Batch, Head, Seq_Len, Seq_Len)")
        print(attention_scores)

        # 4. 乘以 Value
        output = attention_scores @ value
        print("\n--- 步骤 5：注意力权重 乘以 Value (当前头输出) ---")
        print("形状:", output.shape, "-> (Batch, Head, Seq_Len, d_k)")
        print(output)
        print("="*40 + "\n")
        
        return output, attention_scores    

    def forward(self, q, k, v, mask):
        print("\n--- 步骤 1：原始输入流 (Dummy Input) ---")
        print("形状:", q.shape, "-> (Batch, Seq_Len, d_model)")
        print(q)

        print("\n--- 步骤 2：窥探机器内部的权重矩阵 (W_Q 的真实面目) ---")
        print("形状:", self.w_q.weight.shape, "-> (d_model, d_model)")
        print(self.w_q.weight)

        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        print("\n--- 步骤 3：经过 W_Q 线性映射后的 Query (未切分) ---")
        print("形状:", query.shape, "-> (Batch, Seq_Len, d_model)")
        print(query)

        # 【魔法核心】切分多头
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)

        print("\n--- 步骤 3.5：切分并换位后的 Query (多头状态) ---")
        print("形状:", query.shape, "-> (Batch, Head, Seq_Len, d_k)")
        print(query)

        x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # 合并多头
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)
        print("\n--- 步骤 6：多头重新合并后的张量 ---")
        print("形状:", x.shape, "-> (Batch, Seq_Len, d_model)")
        print(x)

        final_out = self.w_o(x)
        print("\n--- 步骤 7：经过 W_O 映射的最终输出 ---")
        print("形状:", final_out.shape, "-> (Batch, Seq_Len, d_model)")
        print(final_out)
        
        return final_out

### 准备数据并执行前向传播
我们将模拟：
* 2 句话 (Batch = 2)
* 每句话 3 个词 (Seq_Len = 3)
* 每个词 4 维向量 (d_model = 4)
* 分 2 个头 (Heads = 2)

**特别注意 Mask 的设计**：
* Batch 0 没有遮挡 `[1, 1, 1]`
* Batch 1 遮挡了最后一个词 `[1, 1, 0]`

In [3]:
# 1. 设定超参数
BATCH_SIZE = 2      
SEQ_LEN = 3         
D_MODEL = 4         
HEADS = 2           
DROPOUT = 0.0       

# 2. 造机器
mha = MultiHeadAttentionBlock(d_model=D_MODEL, h=HEADS, dropout=DROPOUT)

# 3. 捏造假数据
dummy_input = torch.rand(BATCH_SIZE, SEQ_LEN, D_MODEL) 

# 4. 构造包含遮挡逻辑的真实 Mask
real_mask = torch.tensor([
    [[ [1, 1, 1] ]],  # Batch 0: 全是真实词
    [[ [1, 1, 0] ]]   # Batch 1: 最后一个词是空白填充
])

print(f"准备喂入模型的 Mask 形状: {real_mask.shape}")
print(f"Mask 内容:\n{real_mask}\n")

# 5. 一键启动流水线！(在 Jupyter 里直接调用对象即可)
_ = mha(dummy_input, dummy_input, dummy_input, real_mask)

准备喂入模型的 Mask 形状: torch.Size([2, 1, 1, 3])
Mask 内容:
tensor([[[[1, 1, 1]]],


        [[[1, 1, 0]]]])


--- 步骤 1：原始输入流 (Dummy Input) ---
形状: torch.Size([2, 3, 4]) -> (Batch, Seq_Len, d_model)
tensor([[[0.65, 0.40, 0.91, 0.20],
         [0.20, 0.20, 0.95, 0.67],
         [0.98, 0.09, 0.00, 0.11]],

        [[0.16, 0.70, 0.68, 0.92],
         [0.24, 0.16, 0.77, 0.30],
         [0.80, 0.38, 0.79, 0.11]]])

--- 步骤 2：窥探机器内部的权重矩阵 (W_Q 的真实面目) ---
形状: torch.Size([4, 4]) -> (d_model, d_model)
Parameter containing:
tensor([[ 0.38,  0.42, -0.12,  0.46],
        [-0.11,  0.10, -0.24,  0.29],
        [ 0.44, -0.37,  0.43,  0.09],
        [ 0.37,  0.07,  0.24, -0.07]], requires_grad=True)

--- 步骤 3：经过 W_Q 线性映射后的 Query (未切分) ---
形状: torch.Size([2, 3, 4]) -> (Batch, Seq_Len, d_model)
tensor([[[ 0.40, -0.19,  0.56,  0.47],
         [ 0.36, -0.04,  0.49,  0.27],
         [ 0.46, -0.07,  0.41,  0.36]],

        [[ 0.70,  0.16,  0.20,  0.21],
         [ 0.21, -0.11,  0.41,  0.26],
         [ 0.42, -0.21,  0